# COMET Evaluation for Arabic Question Answering Models

This notebook evaluates the semantic quality of the Arabic question-answering models using the COMET metric.

COMET scores are calculated for the RNN, GRU, LSTM, mT5, Qwen2.5, and AraGPT2 experiments and are combined with the BLEU results to support final model selection.

## 1. Setup and Imports

In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu tensorflow-datasets tensorboard
!pip install protobuf==4.25.9
!pip install unbabel-comet pandas

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import pandas as pd
from comet import download_model, load_from_checkpoint

## 2. Load COMET Evaluation Model

In [ ]:
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

## 3. COMET Scoring Function

In [8]:
def calculate_comet(file_name):

  df = pd.read_csv(file_name)

  data = [
      {
          "src": df["src"].iloc[i],
          "mt": df["mt"].iloc[i],
          "ref": df["ref"].iloc[i]
      }
      for i in range(len(df))
  ]

  comet_score = comet_model.predict(
      data,
      batch_size=2,
      gpus=0
  )

  return df["Model"].iloc[0], comet_score.system_score

## 4. Evaluate All Question Answering Models

In [9]:
files = [
    "rnn_tfidf_comet.csv",
    "rnn_cbow_comet.csv",
    "rnn_bert_comet.csv",
    "gru_tfidf_comet.csv",
    "gru_cbow_comet.csv",
    "gru_bert_comet.csv",
    "lstm_tfidf_comet.csv",
    "lstm_cbow_comet.csv",
    "lstm_bert_comet.csv",
    "t5_comet.csv",
    "qwen_comet.csv",
    "gpt_comet.csv",
]


In [10]:
comet_results_list = []

for file in files:

  model_name, comet_score = calculate_comet(file)

  comet_results_list.append({"Model": model_name, "Comet Score": comet_score})

  comet_results = pd.DataFrame(comet_results_list)

  comet_results

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 50/50 [03:02<00:00,  3.65s/it]
INFO:pytorch_lightning.utilities.rank_zero:GP

## 5. BLEU and COMET Final Comparison

In [11]:
bleu_results = pd.DataFrame({
    "Model": ["RNN + TF-IDF",
              "RNN + AraVec CBOW",
              "RNN + AraBERT",
              "GRU + TF-IDF",
              "GRU + AraVec CBOW",
              "GRU + AraBERT",
              "LSTM + TF-IDF",
              "LSTM + AraVec CBOW",
              "LSTM + AraBERT",
              "T5",
              "Qwen",
              "GPT"],

    "BLEU Score": [

        0.000225,
        0.000225,
        0.000255,
        0.000095,
        0.000247,
        0.000720,
        0.000260,
        0.000258,
        0.000209,
        16.067716,
        13.802488,
        5.213261
    ]
})

In [12]:
final_results = pd.merge(bleu_results, comet_results, on="Model")

final_results

,Model,BLEU Score,Comet Score
0,RNN + TF-IDF,0.000225,0.295240
1,RNN + AraVec CBOW,0.000225,0.296160
2,RNN + AraBERT,0.000255,0.327364
3,GRU + TF-IDF,0.000095,0.318316
4,GRU + AraVec CBOW,0.000247,0.322380
5,GRU + AraBERT,0.000720,0.275974
6,LSTM + TF-IDF,0.000260,0.243458
7,LSTM + AraVec CBOW,0.000258,0.243864
8,LSTM + AraBERT,0.000209,0.253719
9,T5,16.067716,0.554376
